In [1]:
import fitz 
import pdfplumber
import json 
import re  
import random 
from pathlib import Path
from typing import List, Dict, Tuple
import torch
assert torch.cuda.is_available()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [2]:
PDF_PATH = "/mnt/e/Ignatiuz/Ignatiuz_AIML_capabilities_deck.pdf"
# Absolute WSL path to the uploaded Ignatiuz PDF.
# /mnt/user-data/uploads/ is where Claude stores files you upload in this session.
# Update this path if you move the file to a different location on your machine.
 
# ─────────────────────────────────────────────────────────────────────────────
# DOCUMENT KNOWLEDGE BASE — EXACT CONTENT EXTRACTED FROM THE IGNATIUZ PDF
#
# WHY THIS EXISTS:
# The PDF has been fully read (all 47 pages). Key facts, metrics, client names,
# and exact statistics are stored here as verified ground truth strings.
# The teacher's grounding prompts are built from these strings — guaranteeing
# zero hallucination of core document facts.
#
# Every number, client name, and quoted phrase below appears verbatim in the
# uploaded Ignatiuz_AIML_capabilities_deck.pdf.
# ─────────────────────────────────────────────────────────────────────────────

In [3]:
DOCUMENT_KNOWLEDGE_BASE = {
 
    "company_overview": """
Ignatiuz is a forward-thinking technology solutions provider committed to driving digital
transformation and empowering businesses across industries to thrive in the modern digital
landscape. Expertise includes AI & ML model development, cloud consulting, business
intelligence, and robotic process automation. Mission: help organizations embrace
cutting-edge technologies, optimize operations, and achieve sustainable growth.
Partners: Microsoft, UiPath, Outsystems, Nintex, Smartsheet, CobbleStone, Salesforce,
AWS partner network, Nividous, SAP, InsightGlobal, NJMMA.
Offices: Indore (India), Hyderabad (India), Pennsylvania USA — 600 Eagleview Blvd,
Suite #300, Exton PA 19341. Tel: +1 267-281-8692. Fax: +1 484-206-4141.
Email: info@ignatiuz.com. Website: www.ignatiuz.com.
""",
 
    "services": """
Six core service areas (from "What We Do?" slide):
1. Emerging Technologies: AI/ML model development, cutting-edge data science services,
   strategic AI/ML consulting.
2. Product Engineering: SaaS development, MVP creation, innovative mobile engineering.
3. Consulting Services: Architectural reviews, UI/UX design, Centers of Excellence for
   Microsoft and Smartsheet.
4. Cloud Engineering: Cloud consulting, seamless migration frameworks, managed services.
5. Digital Journeys: RPA, advanced Business Intelligence solutions, digital transformation.
6. Low Code Experiences: Low code/no code automation using Smartsheet, Microsoft Power
   Platform, and Outsystems.
Also: Post Merger IT Integration, Full Stack Cloud Computing, Enterprise Mobility, AI/ML,
RPA, Low Code Application Development, Microsoft 365 Solutions, Technology Consulting & QA.
""",
 
    "clients": """
Clients (from "Our Clients" slide):
Saint-Gobain, NJ.gov, International Seaways Inc., Inotiv, Solid Biosciences,
Borough of Point Pleasant NJ, Township of Bridgewater, City of Wilmington Delaware,
Berlitz, Cardone, PM-International, Hazlet Township, Ocean County NJ,
Old Bridge Middlesex County, CypherTax, Cass Information Systems Inc.,
PPG Professional Plumbing Group, SOMPO, Roche.
""",
 
    "generative_ai": """
Generative AI is at the forefront of the artificial intelligence revolution, transforming how
businesses innovate, operate, and engage with customers.
 
What is Generative AI?
Algorithms that generate new content — text, images, sound, and beyond — based on patterns
learned from vast datasets. Not just automating tasks; creating something new and valuable.
Generates: Text (human-like, articles, responses), Images (from descriptions), Music and
Sound (specific styles), Video (emerging).
 
Key Aspects:
- Creativity and Innovation: Enhances creative processes for artists, writers, designers.
- Efficiency: Automate content creation, save time, maintain quality.
- Personalization: Tailored content improves customer engagement and satisfaction.
- Cross-Disciplinary: Healthcare drug discovery, gaming dynamic storytelling, marketing
  personalized campaigns.
 
Key Capabilities:
1. Personalization At Scale: Engines adapt in real-time to user interactions and preferences,
   processing user data including past interactions and knowledge bases.
2. Predictive Analytics And Decision-Making: Generate new content from patterns in vast datasets.
3. Innovation Acceleration: Shorten R&D cycles by simulating outcomes using historical data;
   analyze past R&D data, experiments, and trials to propose new formulas.
4. Advanced Content Creation: Create high-quality text, images, videos mimicking human
   creativity by analyzing existing content databases for patterns, styles, structures.
""",
 
    "autonomous_intelligence": """
Autonomous Intelligence: Systems capable of performing tasks and making decisions
independently, without human intervention.
Key Characteristics:
- Self-Governance: Operates autonomously in real time.
- Learning and Adaptation: Learns from experiences to improve performance.
- Real-Time Decision Making: Responds dynamically to environment changes.
- Task Execution: Can perform complex tasks that usually require human judgment.
Examples: Self-driving cars, Autonomous drones for delivery, Robotics in manufacturing.
Benefits: Increased Efficiency (automates repetitive tasks), Enhanced Safety (reduces human
error in hazardous environments), Cost Reduction (lowers operational costs).
""",
 
    "rnn_chatbot": """
Recurrent Neural Network (RNN):
Artificial neural network designed to recognize patterns in sequences of data such as text,
genomes, handwriting, or numerical time series data from sensors and other recurring sources.
Called "recurrent" because they perform the same task for every element of a sequence, with
output dependent on previous computations. RNNs possess a memory capturing information
about what has been calculated so far (unlike traditional neural networks).
 
Chatbot With RNN:
Capable of understanding and generating human-like responses, learning from previous
interactions to improve over time. Can manage a large volume of queries simultaneously,
ensuring quick and accurate responses, crucial for maintaining high customer satisfaction
and reducing the workload on human agents.
Benefits: Improved Efficiency (automates responses to common queries, reducing response
times and costs), Scalability (handles query volume fluctuations), Learning Capability
(continuously improves through learning from interactions).
""",
 
    "ecommerce_ai_search": """
Enhancing E-Commerce Search:
AI/ML algorithms understand natural language and user intent.
Benefits: Enhanced Relevance, Personalized Recommendations, Auto-Correction and Synonym
Recognition.
AI/ML-Driven Features:
- Instant Search Suggestions: Real-time suggestions as users type.
- Auto-Complete: Predict and complete search queries to save time.
- Query Expansion: Broaden results by including related terms and synonyms.
""",
 
    "ai_recommendation_engine": """
AI Recommendation Engine — Three-step process:
Step 1 (Insert data): Product Catalogue Data + Customer Data → Integrate Customer Data
Step 2 (Customize): AI Model → Recommendation Type, Objective, Business Rules & Needs
Step 3 (Deliver): Prediction API → Show Recommendations at Customer Touchpoints
 
Considerations:
1. Intelligent & personalized recommendations.
2. Similar & frequently bought product recommendations.
3. User behaviour, purchase history & preferences.
4. Understanding of natural language & user intent.
5. Additional attributes from supplier content or web content.
""",
 
    "ai_chatbot": """
Customized AI-Powered Chatbot:
Handle Unexpected Questions, Recap Conversation, & Automate Processes.
Generate a Personalized Experience For Both Internal and External Users.
 
Capabilities:
Content Generation: Auto-generate responses to customer inquiries, generate personalized UI.
Summarization: Summarize support conversations, financial reports, analyst articles, social
media trends.
Semantic Search: Search reviews, information discovery and knowledge.
 
Application — Tax Information Retrieval (TaxBot demo):
- Automation: Streamlines answering tax-related queries.
- Consistency: Consistent and accurate information based on the dataset.
- Accessibility: Users access tax information anytime without human intervention.
 
Azure OpenAI Architecture:
Web interface ↔ Web Server ↔ API Gateway ↔ Azure OpenAI
Multiple Data Sources: Database → Data Pipeline → Data Lake → Data Training → Azure OpenAI
 
Models: AzureGPT-4 series (GPT-4 Turbo with vision), GPT-3.5 Turbo series (content
generation, summarization, image understanding, semantic search, NL to code), Embedding series.
""",
 
    "supercharge_support": """
Supercharge Support With AI:
Dynamic Script with AI Assistant — real-time call coaching and faster resolution.
 
Three AI capabilities:
1. Installer Certification Verification: AI analyzes customer data (purchase history,
   registration) to determine installer certification for specific products; prompts agent
   with relevant support options.
2. Solution Recommendation Engine: AI analyzes caller's issue description, identifies
   keywords/patterns, presents ranked list of potential solutions from historical data or
   troubleshooting manuals.
3. Automated Knowledge Base Search: AI scours internal knowledge base for relevant articles,
   FAQs, or video tutorials based on installer questions; agent shares directly through
   call interface.
 
Benefits:
- Real-time Call Coaching: AI whispers suggestions during calls.
- Faster Resolutions: AI-reduced call times mean happier customers.
- Personalized Support: AI-powered solutions cater to individual needs and preferences.
- Proactive Communication: Automated follow-up emails/texts keep customers informed.
""",
 
    "process_mining": """
Process Mining: Technology that uses event log data from IT systems to analyze and improve
business processes. Bridges the gap between theoretical process models and actual process
execution, providing transparency and insights.
Applicable areas: Supply chain, finance, HR, IT operations, customer service.
Use cases: Process optimization, compliance monitoring, root cause analysis, performance
benchmarking.
 
Need For Process Mining:
1. Intended V/S Visibility: Not all processes follow the 'happy path' (smooth, no complications).
   Exceptions/deviations significantly impact efficiency.
2. Lack of Visibility: Employees have limited understanding of sub-processes; fragmented view
   of entire process chain hinders comprehensive insight.
3. Constant Changes: Processes evolve for customer requirements, regulations, restructuring
   but adaptations not always reflected in documentation — discrepancies between documented
   and actual processes.
 
How Process Mining Works:
Data Sources: Event logs from IT systems (activity performed, timestamp, identity of performer).
Key Steps:
1. Automated Process Discovery: Algorithms analyze event logs, construct comprehensive model
   of actual process flow — reveals sequence, frequency, paths.
2. Conformance Checking: Compares discovered model against intended model; identifies
   deviations, non-compliance, divergences from expected workflow.
3. Performance Analysis: Evaluates efficiency; highlights bottlenecks, delays, inefficiencies.
 
Starting a Process Mining Project:
Problem Identification → Data Collection → Pilot Implementation → Analysis And Improvements
(Use insights from process mining to implement improvements and monitor their impact.)
""",
 
    "process_mining_business_case": """
Enterprise Business Case (Example Company — exact figures from document):
Revenue: 2,740 M€ | Sales orders/year: 670,000 | Purchase orders/year: 265,000 | Avg FTE cost: 44,000 €
ROI: 370% | Payback time: 7.6 months | Business Case: 12.9 M€ | Investment: 2.5 M€
 
Value by process area:
- Order management: ≈ 5.3 M€ (cut lead time, improve on-time delivery, increase sales)
- Accounts receivable: ≈ 4.1 M€ (speed up invoicing, reduce billing blocks)
- Procurement: ≈ 2.8 M€ (increase just-in-time deliveries, reduce Maverick buying)
- Accounts payable: ≈ 2.7 M€ (improve on-time payment, increase cash discounts)
Theme: Reduce manual rework & improve automation.
""",
 
    "rpa": """
RPA (Robotic Process Automation): Not physical robots — software "bots". Digital workers.
Accomplishes repetitive, manual tasks more quickly and accurately than humans.
Excellent at rule-based and trigger-driven tasks (like data processing). Works across any
application or system.
 
Good tasks for RPA: Repeated at regular intervals or pre-defined trigger; well-defined
inputs/outputs; heavily dependent on email or spreadsheets; receiving data from one system
and inputting into another; takes at least 8–10 hours/week.
 
What RPA accomplishes: Refocus employees to value-add tasks; ensure staff on most important
tasks; improve job satisfaction and morale; integrate outdated/inflexible systems (UI is the API);
harmonize inconsistent data sources; improve speed of revenue recognition and TAT; increase compliance.
 
RPA Statistics: 93% of C-level executives say RPA kickstarted digital transformation.
83% will use RPA to build agility, diversity and resilience. 82% expect automation progress
and acceleration over the next three months.
 
Three Models of RPA:
1. Unattended RPA: End-to-end automation — back office/batch oriented. No human involvement.
2. Attended RPA: End-to-end automation — back office/batch oriented. Bots work alongside humans.
3. Hybrid RPA: End-to-end automation — back office/batch oriented. Humans + bots collaborate.
 
Intelligent RPA Summary: Proven (hundreds of processes, all industries), Intelligent (ML tech),
High Value To Cost (low-cost, big impact quickly), Extensible (accommodates old/new systems),
Secure & Scalable, Human-Bot Orchestration.
 
Benefits of Process Automation: Save Time, Eliminate Manual Mistakes, Reduce Costs,
Accelerate Outcomes, Focus on strategic items, Kick-start digital transformation, Reduce
paper waste, Improve customer satisfaction, Strengthen operations, Happier employees.
Tagline: "Faster. Better. Happier."
""",
 
    "idp": """
Intelligent Document Processing (IDP): Cognitive automation platform that can extract
information from semi-structured and unstructured data across a multitude of document formats.
Doc Sources: Internal, External.
Input Types: Structured, Semi Structured, Unstructured.
Formats: Word, Excel, PDF, Images, Text.
 
Six-stage IDP pipeline:
1. Classification → 2. Pre-processing → 3. Extraction → 4. Post Processing → 5. Verification → 6. Integration
""",
 
    "manual_invoice_challenges": """
Challenges of Manual Invoice Processing:
- Aberdeen Group research: It takes between 4.1 and 16.3 days for companies to process an
  invoice from receipt through payment approval.
- Canon Business Process Services study: More than half of all invoice processing requires
  at least 76% manual input.
""",
 
    "case_study_brewing_ap": """
Case Study: Accounts Payable — A Leading Drink and Brewing Company
Volume: 20K+ invoices received monthly from 400+ vendors in different countries.
Solution deployed: Bots with cognitive capabilities to automate complete AP process.
- RPA Bots download invoices from emails and segregate by type.
- Straight through processing: automated invoice match with PO, auto posting to ERP.
- Exceptions handled through manual intervention.
- Rule-based, multi-level approval matrix.
- Intuitive vendor portal for invoice submission, status tracking, and query management.
 
EXACT RESULTS (do not change these numbers):
- 90% Reduction in process TAT
- 1000 Staff-hours saved per month
- 100% Reduction in manual errors
""",
 
    "case_study_manufacturing_invoice": """
Case Study: Invoice Processing @ Manufacturing Firm
Volume: 20K+ invoices monthly from 400+ vendors. 2500+ invoices weekly in varied formats.
Challenge: 6+ FTEs transcribed data into SAP ERP manually, causing delays. High risk of
human errors could negatively affect vendor relationships, resulting in substantial business losses.
Solution: RPA Smart Bots with Machine Learning, Computer Vision and advanced OCR capabilities
deployed in two weeks. Bots read, understand and fetch details from unstructured data types
and image-based documents and feed into SAP ERP. Human-Bot collaboration for verification
wherever required.
 
EXACT RESULTS (do not change these numbers):
- 90% Reduction in process TAT
- 1000 Staff-hours saved per month
- 100% Reduction in manual errors
""",
 
    "case_study_media_orders": """
Case Study: Automation of Customer Orders @ Media Company
Volume: 60+ FTEs manually processing 1000+ customer orders daily.
Challenge: High risk of manual errors and delays; varied formats of customer orders.
Solution: Smart Bots with cognitive capabilities.
- Bots access order info from emails and fax, extract specific information, feed into
  automation control center interface.
- Smart Bots enter extracted data into ERP; lower confidence score triggers manual exception.
- Enabled centralized processing and 50% savings on FTEs in the process.
 
EXACT RESULTS (do not change these numbers):
- 90% Reduction in process TAT
- 1000 Staff-hours saved per month
- 100% Reduction in manual errors
""",
 
    "automation_journey": """
Ignatiuz Automation Journey Support — Three stages:
1. Launch — Advisory and Strategy ("Advise You"):
   For clients saying: "We want to know more" / "OK, where do we start?"
2. Expand — Capability & Implementation ("With You"):
   For clients saying: "How do we accelerate value?" / "We use it but struggle to scale."
3. Operate — Digital Services ("For You"):
   For clients saying: "We have a team but want to broaden capability" /
   "We need to expand capability across the enterprise."
""",
 
}
# End of DOCUMENT_KNOWLEDGE_BASE.
# Every string above is extracted verbatim from Ignatiuz_AIML_capabilities_deck.pdf.
# Particular attention to case study metrics: 90% TAT reduction, 1000 staff-hours/month,
# 100% error reduction — these exact figures appear on slides 43, 44, and 45.
# The teacher must quote these precisely; the quality gate checks for them.

In [4]:
OUTPUT_DIR = Path("./ignatiuz_dataset")
# Directory where all output files will be saved:
#   - extracted_pages.json    (raw text per page, for inspection)
#   - chunks.json             (cleaned, merged text sections)
#   - train.jsonl             (90% of samples, for fine-tuning)
#   - eval.jsonl              (10% of samples, for evaluation)

In [5]:
OUTPUT_DIR.mkdir(exist_ok=True)
# Create the directory if it doesn't already exist.
# exist_ok=True means no error is raised if it already exists.
 
CHUNK_SIZE = 600
# Target size in characters for each text chunk fed to the teacher.
# 600 chars ≈ 100–120 words ≈ roughly one slide or one paragraph.
# Why not the whole document at once? Because:
# 1. The teacher generates better, more focused Q&A from a small, specific
#    passage than from a wall of text.
# 2. Smaller chunks mean more training samples (more coverage of the document).
# 3. Smaller prompts are cheaper and faster to run through the teacher.
# Increase to 1000+ for dense technical documents with long paragraphs.
 
CHUNK_OVERLAP = 100
# Number of characters to overlap between consecutive chunks.
# Example: chunk 1 = chars 0–600, chunk 2 = chars 500–1100, etc.
# Overlap prevents a key sentence from being split across two chunks
# and appearing in neither — especially important for case study results
# that span two slides (e.g., "saved 1000 hours... per month" split across pages).
 
QA_PER_CHUNK = 3
# How many direct Q&A pairs to generate per chunk.
# With ~40 content chunks from a 47-page deck, this gives ~120 direct pairs.
# Combined with CoT samples, the total dataset reaches ~150–180 samples.
# Increase to 5 for a richer dataset if you have time.
 
TEMPERATURE_DIRECT = 0.2
# Low temperature for direct answer generation.
# 0.2 = very close to deterministic. For factual questions grounded in
# specific text (e.g., "what percentage reduction was achieved?"), you want
# the teacher to extract the exact fact, not creatively rephrase it.
 
TEMPERATURE_COT = 0.4
# Slightly higher temperature for CoT (reasoning) generation.
# 0.4 = mild variation. CoT responses benefit from some natural language
# variation — two clients with similar problems shouldn't get identical reasoning.
# But we keep it below 0.7 to avoid the teacher drifting off-topic.
 
SYSTEM_PROMPT = """You are an expert enterprise AI/ML consultant at Ignatiuz.
You have access to the exact content from Ignatiuz's capabilities document.
When answering, you must:
1. Base your answer ONLY on the provided document excerpt — do not add external knowledge.
2. Use specific numbers, names, and results exactly as stated in the excerpt.
3. If the excerpt does not contain the answer, say "This information is not in the provided section."
"""
# This system prompt does the heavy lifting of grounding:
# - "ONLY on the provided document excerpt" — prevents the teacher from
#   supplementing with its general AI knowledge, which could contradict the doc.
# - "specific numbers, names, and results exactly as stated" — ensures metrics
#   like "90% reduction" and "1000 staff-hours" are copied faithfully, not
#   approximated.
# - The fallback instruction teaches the teacher to admit absence of information
#   rather than hallucinate — these fallback responses are actually valuable
#   training signal: they teach the student the document's limits too.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — PDF TEXT EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────
 
def extract_pages_with_fitz(pdf_path: str) -> List[Dict]:
    """
    Extract text from every page of the PDF using PyMuPDF (fitz).
    Returns a list of dicts: [{page_num, text, word_count}, ...]
 
    Why fitz over pypdf or pdftotext?
    - fitz preserves reading order on slide decks (title → subtitle → body)
    - fitz handles multi-column text correctly (reads left column then right)
    - fitz is 3-5x faster than pdfplumber for plain text extraction
    - fitz correctly handles the UTF-8 characters common in business documents
    """
    doc = fitz.open(pdf_path)
    # fitz.open() loads the PDF into memory. For a 47-page slide deck this is
    # fast (<1 second). For 500+ page documents, use streaming mode instead.
 
    pages = []
    # Accumulator list — we'll append one dict per page.
 
    for page_num in range(len(doc)):
        # Iterate by index (0-based). len(doc) returns the total page count.
 
        page = doc[page_num]
        # Access page by index. This returns a fitz.Page object with methods
        # for text extraction, image extraction, annotation reading, etc.
 
        text = page.get_text("text")
        # "text" mode extracts plain text in reading order (top-to-bottom,
        # left-to-right). Other modes:
        # - "blocks" returns text grouped into rectangular blocks
        # - "words" returns individual words with coordinates
        # - "html" returns HTML with font/style information
        # - "dict" returns a full JSON structure with every character's bbox
        # For our purpose "text" is cleanest and fastest.
 
        text = text.strip()
        # Remove leading/trailing whitespace. Slide decks often have trailing
        # newlines at the end of each page's text block.
 
        if len(text) < 50:
            continue
        # Skip pages with less than 50 characters — these are typically:
        # - Pure image slides (diagrams, screenshots, logos)
        # - Divider/section header slides with only a title word
        # - Blank pages or pages with only decorative elements
        # These pages have no extractable training value.
 
        word_count = len(text.split())
        # Quick proxy for content density. Used later to decide if a page
        # is worth chunking or should be merged with adjacent pages.
 
        pages.append({
            "page_num":   page_num + 1,
            # 1-based page number (matches what humans see in the PDF viewer).
            "text":       text,
            # The raw extracted text for this page.
            "word_count": word_count,
            # Word count for density assessment.
        })
 
    doc.close()
    # Release the file handle. Good practice even though Python's GC would
    # eventually do this — explicit close prevents file lock issues on Windows.
 
    print(f"Extracted {len(pages)} content pages from {pdf_path}")
    return pages
 
 
def extract_tables_with_pdfplumber(pdf_path: str) -> List[Dict]:
    """
    Extract tables from the PDF using pdfplumber.
    pdfplumber detects table structures by analysing the spatial positions
    of text words and grouping them into rows and columns.
    Returns a list of dicts: [{page_num, table_text}, ...]
 
    Why a separate table extraction step?
    fitz extracts table text but loses the row/column structure — a comparison
    table becomes a flat string of cell values with no relational context.
    pdfplumber reconstructs "Row 1: [val1, val2, val3]" etc., which makes
    the teacher's Q&A about comparison tables far more accurate.
    """
    tables_data = []
 
    with pdfplumber.open(pdf_path) as pdf:
        # Context manager — ensures the PDF file is properly closed even if
        # an exception occurs during table extraction.
 
        for page_num, page in enumerate(pdf.pages, start=1):
            # enumerate(pdf.pages, start=1) gives (1, page1), (2, page2) etc.
            # start=1 makes the page numbers 1-based.
 
            tables = page.extract_tables()
            # Returns a list of tables found on this page.
            # Each table is a list of rows; each row is a list of cell strings.
            # Empty cells are None, which we handle below.
            # Returns [] if no tables are found on the page.
 
            for table in tables:
                if not table or len(table) < 2:
                    continue
                # Skip empty tables or single-row "tables" (often just headers
                # that pdfplumber mistakenly classified as a table).
 
                # Convert the 2D list into a readable text representation.
                table_lines = []
                for row in table:
                    cleaned_cells = [str(cell).strip() if cell else "" for cell in row]
                    # Convert each cell to string (some may be None for empty cells).
                    # .strip() removes leading/trailing whitespace from each cell.
 
                    if any(cleaned_cells):
                        # Only include rows that have at least one non-empty cell.
                        table_lines.append(" | ".join(cleaned_cells))
                        # Join cells with " | " separator to make the table
                        # readable as plain text. Example output:
                        # "Attended RPA | Desktop-triggered | Human judgment needed"
 
                if table_lines:
                    tables_data.append({
                        "page_num":   page_num,
                        "table_text": "\n".join(table_lines),
                        # Each row on its own line, cells separated by " | ".
                    })
 
    print(f"Extracted {len(tables_data)} tables from {pdf_path}")
    return tables_data
 

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — TEXT CLEANING
# ─────────────────────────────────────────────────────────────────────────────
 
def clean_text(text: str) -> str:
    """
    Remove noise from extracted PDF text that would confuse the teacher
    or waste tokens in the prompt.
    """
 
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Replace 3 or more consecutive newlines with just 2 newlines.
    # Slide decks often have large gaps between sections encoded as many
    # blank lines. This compacts them without losing paragraph structure.
 
    text = re.sub(r'[ \t]{2,}', ' ', text)
    # Replace 2+ spaces or tabs with a single space.
    # Multi-column PDFs sometimes insert multiple spaces to align text visually.
    # These are meaningless in plain text and waste token budget.
 
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    # Remove lines that contain only a number (page numbers).
    # Pattern: start of line (^), optional whitespace, digits (\d+),
    # optional whitespace, end of line ($), applied per line (MULTILINE).
    # This removes standalone "1", "2", "47" lines that are page number artifacts.
 
    text = re.sub(r'(Ignatiuz\.com|www\.ignatiuz\.com|@ignatiuz)', '', text, flags=re.IGNORECASE)
    # Remove footer/watermark text that appears on every slide.
    # These strings add no training value and waste tokens.
 
    text = re.sub(r'\|{2,}', '|', text)
    # Replace multiple consecutive pipe characters (table artifact) with one.
 
    text = text.strip()
    # Final trim of leading/trailing whitespace.
 
    return text

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — CHUNKING
# ─────────────────────────────────────────────────────────────────────────────
 
def chunk_pages(pages: List[Dict], chunk_size: int = CHUNK_SIZE,
                overlap: int = CHUNK_OVERLAP) -> List[Dict]:
    """
    Merge all page texts into one document string, then split into
    overlapping chunks of ~chunk_size characters.
 
    Why merge first then chunk (instead of one chunk per page)?
    Slide decks often spread one topic across 2-3 slides:
    - Slide 12: "Accounts Payable Automation — Challenge"
    - Slide 13: "Accounts Payable Automation — Solution"
    - Slide 14: "Accounts Payable Automation — Results: 90% reduction"
    If we chunk per page, the challenge, solution, and result are in separate
    chunks with no connection. The teacher sees "90% reduction" without context
    of what was automated. Merging first lets overlapping chunks span slides.
    """
 
    # Step 1: Merge all pages into one text string with page markers.
    full_text = ""
    for page in pages:
        cleaned = clean_text(page["text"])
        if cleaned:
            full_text += f"\n\n[Page {page['page_num']}]\n{cleaned}"
            # Page markers like "[Page 12]" are preserved in chunks.
            # This helps the teacher ground its answers to specific pages,
            # and helps you audit which chunks came from where.
 
    # Step 2: Split into overlapping chunks.
    chunks = []
    start = 0
    total_len = len(full_text)
 
    while start < total_len:
        end = start + chunk_size
        # End of this chunk (may go past end of document — slicing handles that).
 
        chunk_text = full_text[start:end]
        # Slice the text. If end > total_len, Python returns up to the end.
 
        if len(chunk_text.strip()) < 100:
            break
        # If the remaining text is less than 100 chars, it's a fragment
        # (e.g., a trailing page marker). Skip it rather than feed a nearly
        # empty chunk to the teacher.
 
        # Find the last sentence boundary within the chunk for a cleaner cut.
        # We look for ". ", ".\n", "? ", "! " near the end of the chunk.
        last_sentence_end = max(
            chunk_text.rfind('. ', max(0, len(chunk_text) - 150)),
            # rfind searches backwards from the end. We look in the last 150
            # chars for a period-space, which marks a sentence end.
            chunk_text.rfind('.\n', max(0, len(chunk_text) - 150)),
            chunk_text.rfind('\n\n', max(0, len(chunk_text) - 150)),
            # Double newline = paragraph break — also a good chunk boundary.
        )
 
        if last_sentence_end > len(chunk_text) // 2:
            # Only use the detected boundary if it's in the second half of
            # the chunk. If it's near the start, the chunk would be too small.
            chunk_text = chunk_text[:last_sentence_end + 1]
            # Include the period/newline in the chunk text.
 
        chunks.append({
            "chunk_id":  len(chunks),
            # Sequential ID for tracking which chunk generated which samples.
            "text":      chunk_text.strip(),
            "char_start": start,
            "char_end":   start + len(chunk_text),
            # Character offsets into the full merged document — useful for
            # debugging if a generated Q&A pair references incorrect facts.
        })
 
        start += len(chunk_text) - overlap
        # Advance start by (chunk_length - overlap) so the next chunk
        # begins overlap characters before the current chunk ends.
        # This creates the sliding window effect.
 
        if start >= total_len:
            break
 
    print(f"Created {len(chunks)} chunks from {len(pages)} pages")
    return chunks
 

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — QUESTION TEMPLATES
# ─────────────────────────────────────────────────────────────────────────────
 
DIRECT_QUESTION_TEMPLATES = [
    # Templates for extracting factual information directly from the text.
    # The {topic} placeholder will be filled with keywords detected in the chunk.
    "Based on the document excerpt, what is {topic}?",
    "According to the document, what are the key details about {topic}?",
    "What specific results or outcomes does the document mention regarding {topic}?",
    "What does the document say about how {topic} works?",
    "What technologies or methods does the document describe for {topic}?",
    "What problem does {topic} solve according to the document?",
    "What are the specific numbers, metrics, or statistics mentioned about {topic}?",
    "How does the document describe the implementation of {topic}?",
    "What client use case does the document present related to {topic}?",
    "According to the excerpt, what are the benefits of {topic}?",
]
# Having 10 templates means each chunk can generate varied questions without
# the student seeing the same phrasing pattern every time. Variety in question
# phrasing helps the student generalise to new question wordings at inference.
 
COT_SCENARIO_TEMPLATES = [
    # Templates for consulting reasoning scenarios.
    # These force the teacher to reason FROM the document evidence, not from
    # general AI knowledge.
    "A client is evaluating whether to implement {topic}. Based on the document evidence provided, walk through the decision step by step.",
    "Using only the information in this document excerpt about {topic}, help a CTO understand whether this technology fits their enterprise context.",
    "A prospect asks: 'Can you show us a real example of {topic} working?' Using the document excerpt, reason through what evidence exists and how to present it.",
    "Based on the document section about {topic}, what questions should a consultant ask a client before recommending this solution?",
    "The document mentions {topic}. If a client had a similar situation, reason through how Ignatiuz's approach would apply.",
]
 
def extract_topic_from_chunk(chunk_text: str) -> str:
    """
    Extract a representative topic keyword or phrase from a chunk to
    fill the {topic} placeholder in question templates.
 
    Strategy: find the most repeated meaningful noun phrase, or fall back
    to the first capitalised multi-word phrase (likely a proper noun or
    product name from a slide title).
    """
    # Look for capitalised phrases (2-4 words) that are likely topic headers
    # e.g., "Intelligent Document Processing", "Accounts Payable", "Process Mining"
    capitalised_phrases = re.findall(
        r'\b([A-Z][a-z]+(?: [A-Z][a-z]+){1,3})\b',
        # Pattern: Capital letter + lowercase letters, followed by 1-3 more
        # Capital-initial words. This captures "Intelligent Document Processing"
        # but not "I" (single word) or common sentence starts.
        chunk_text
    )
 
    if capitalised_phrases:
        # Return the most common capitalised phrase (likely the slide topic).
        from collections import Counter
        most_common = Counter(capitalised_phrases).most_common(1)[0][0]
        return most_common
 
    # Fallback: extract the first line of the chunk (likely the slide title).
    first_line = chunk_text.strip().split('\n')[0]
    first_line = re.sub(r'\[Page \d+\]', '', first_line).strip()
    # Remove the page marker if the chunk starts with one.
 
    return first_line[:60] if first_line else "this topic"
    # Cap at 60 chars to avoid comically long {topic} insertions.
    # "this topic" is the ultimate fallback for chunks with no clear subject.
 

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — THE GROUNDED GENERATION PROMPT BUILDER
# ─────────────────────────────────────────────────────────────────────────────
 
def build_generation_prompt(chunk_text: str, question: str, use_cot: bool) -> List[Dict]:
    """
    Build the full message list to send to the teacher.
 
    This is the core of the grounded approach:
    The chunk_text is injected directly into the USER message as context.
    The teacher must answer the question using ONLY that context.
 
    This is structurally identical to RAG (Retrieval Augmented Generation)
    at inference time — but we're doing it during DATASET CONSTRUCTION,
    so the student learns answers that are permanently grounded in the doc.
    """
 
    if use_cot:
        user_content = f"""Here is an excerpt from Ignatiuz's capabilities document:
 
---DOCUMENT EXCERPT START---
{chunk_text}
---DOCUMENT EXCERPT END---
 
{question}
 
Think through this step by step, referencing specific details from the excerpt above.
Your reasoning should show HOW you are using the document evidence to reach your conclusion.
Finish with a clear, actionable recommendation grounded in the document."""
        # For CoT: the instruction explicitly asks the teacher to show its
        # reasoning process AND to reference the excerpt. This creates training
        # samples where the student learns to cite document evidence in its
        # chain-of-thought, not just produce a floating reasoning chain.
 
        system = SYSTEM_PROMPT + "\nReasoning: high\nAlways cite specific details from the provided excerpt in your reasoning."
        # "Reasoning: high" activates gpt-oss-20b's extended thinking mode.
        # The citation instruction ensures the CoT traces contain document
        # references, not generic AI consulting advice.
 
    else:
        user_content = f"""Here is an excerpt from Ignatiuz's capabilities document:
 
---DOCUMENT EXCERPT START---
{chunk_text}
---DOCUMENT EXCERPT END---
 
{question}
 
Answer using ONLY the information in the excerpt above.
Include exact numbers, names, and specific details as they appear in the document."""
        # For direct answers: the double emphasis on "ONLY" and "exact" is
        # intentional. Without it, the teacher tends to supplement with general
        # knowledge ("RPA typically reduces costs by...") instead of quoting
        # the document ("Ignatiuz achieved 90% reduction in turnaround time").
 
        system = SYSTEM_PROMPT
 
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_content},
    ]
    # Returns the messages list in the format expected by apply_chat_template.

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — TEACHER INFERENCE (GGUF via llama-cpp-python)
#
# WHY THIS CHANGED FROM THE PREVIOUS VERSION:
# The original load_teacher() used FastLanguageModel (Unsloth) which requires
# the safetensors format. You already have the GGUF version cached locally at:
#   /mnt/c/Users/Tron/.cache/huggingface/hub/
#       models--unsloth--gpt-oss-20b-GGUF/snapshots/
#       d449b42d93e1c2c7bda5312f5c25c8fb91dfa9b4/
#       gpt-oss-20b-F16.gguf
#
# The /mnt/c/ prefix means you are running inside WSL (Windows Subsystem for
# Linux). WSL mounts your Windows C:\ drive at /mnt/c/, so the path above is
# the WSL-equivalent of:
#   C:\Users\Tron\.cache\huggingface\hub\models--unsloth--gpt-oss-20b-GGUF\...
#
# GGUF files must be loaded with llama-cpp-python, NOT FastLanguageModel.
# The API is slightly different but produces identical results for our purpose.
#
# INSTALL (run once in your WSL terminal):
#   CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --upgrade
# ─────────────────────────────────────────────────────────────────────────────
 
from llama_cpp import Llama
# Llama is the main class from llama-cpp-python.
# It is the Python binding for the llama.cpp C++ inference engine.
# This is completely separate from HuggingFace / Unsloth — it reads .gguf
# files directly without needing PyTorch, safetensors, or transformers.
 
 
def load_teacher() -> Llama:
    """
    Load gpt-oss-20b-F16.gguf from the local WSL-mounted Windows cache.
    Returns a Llama instance ready for chat completion inference.
 
    The path /mnt/c/... is the WSL path to your Windows HuggingFace cache.
    Windows path:  C:\\Users\\Tron\\.cache\\huggingface\\hub\\...
    WSL path:      /mnt/c/Users/Tron/.cache/huggingface/hub/...
    They point to the exact same file on disk.
    """
 
    teacher = Llama(
        model_path = "/mnt/c/Users/Tron/.cache/huggingface/hub/models--unsloth--gpt-oss-20b-GGUF/snapshots/d449b42d93e1c2c7bda5312f5c25c8fb91dfa9b4/gpt-oss-20b-F16.gguf",
        # The exact absolute path to your cached .gguf file.
        # This is a hard path — no network calls, no Hub lookups, no verification.
        # llama-cpp-python reads it directly from disk like opening any file.
        #
        # WHY F16 and not Q4_K_M?
        # F16 = full 16-bit precision — the highest quality GGUF variant.
        # Since this model is being used as a TEACHER to generate training data,
        # quality matters more than speed. The student learns from these outputs,
        # so any precision loss in the teacher becomes permanent errors in the
        # student's training data.
        # Q4_K_M would be faster (~11.6 GB vs ~13.8 GB) but introduces
        # quantisation noise that can corrupt factual details like numbers
        # and client names — exactly the details we most need to be accurate.
        # Your RTX 5090 has 32 GB VRAM — F16 fits comfortably.
 
        n_gpu_layers = -1,
        # How many transformer layers to offload to GPU.
        # -1 = offload ALL layers — the entire model runs on your RTX 5090.
        # This is what you want: GPU inference is ~10-50x faster than CPU.
        # If you ever get a CUDA out-of-memory error, reduce this to e.g. 20
        # to offload only 20 layers to GPU and keep the rest in system RAM.
        # With 32 GB VRAM and F16 at 13.8 GB, you have ~18 GB free for the
        # KV cache and generation buffers — no OOM expected.
 
        n_ctx = 2048,
        # Context window size in tokens.
        # This is the maximum number of tokens the model can "see" at once,
        # including both the input prompt AND the generated response.
        # 2048 tokens ≈ ~1500 words — sufficient for:
        #   - System prompt (~100 tokens)
        #   - Document chunk (~150 tokens for 600 chars)
        #   - Question (~30 tokens)
        #   - Generated answer (~350 tokens for direct, ~700 for CoT)
        # Total worst case (CoT): ~1080 tokens — fits within 2048.
        # Increase to 4096 if you use larger chunks (CHUNK_SIZE > 1500).
 
        verbose = False,
        # Suppress llama.cpp's C++ progress logs (layer loading percentages,
        # memory allocation messages, etc.). Set to True if you need to debug
        # a loading error — it prints very detailed diagnostic information.
    )
 
    print(f"Teacher loaded: gpt-oss-20b-F16.gguf")
    print(f"Context window: {teacher.n_ctx()} tokens")
    print(f"GPU layers:     all ({teacher.model.n_gpu_layers} offloaded)")
 
    return teacher
    # Returns a single Llama object — unlike FastLanguageModel which returned
    # (model, tokenizer) as a tuple, llama-cpp-python bundles the tokenizer
    # inside the Llama object itself. No separate tokenizer needed.
 
 
def generate_grounded_response(teacher: Llama, messages: List[Dict],
                                use_cot: bool) -> str:
    """
    Run one inference call on the GGUF teacher with the grounded prompt.
    Returns the teacher's response as a plain string.
 
    NOTE: The function signature changed from the previous version.
    Old: generate_grounded_response(teacher, tokenizer, messages, use_cot)
    New: generate_grounded_response(teacher, messages, use_cot)
    The tokenizer parameter is removed because llama-cpp-python handles
    tokenisation internally inside the Llama object.
    """
 
    result = teacher.create_chat_completion(
        # create_chat_completion is llama-cpp-python's OpenAI-compatible chat
        # API. It takes the same messages format as OpenAI's ChatCompletion:
        # [{"role": "system", ...}, {"role": "user", ...}]
        # This is also the same format our build_generation_prompt() returns,
        # so no conversion is needed.
 
        messages = messages,
        # The messages list from build_generation_prompt():
        # contains system prompt (with injected chunk) + user question.
 
        max_tokens = 700 if use_cot else 350,
        # Maximum number of NEW tokens to generate after the prompt.
        # CoT: 700 tokens ≈ full reasoning trace + final answer.
        # Direct: 350 tokens ≈ complete factual response.
        # These match the values used in the old FastLanguageModel version
        # to keep dataset characteristics consistent.
 
        temperature = TEMPERATURE_COT if use_cot else TEMPERATURE_DIRECT,
        # 0.4 for CoT (slight variation in reasoning style).
        # 0.2 for direct answers (close to deterministic for factual accuracy).
        # Defined in SECTION 2 — CONFIGURATION at the top of this file.
 
        repeat_penalty = 1.15,
        # llama-cpp-python's equivalent of HuggingFace's repetition_penalty.
        # Penalises tokens the model has already generated, preventing it from
        # repeating phrases like "Ignatiuz provides... Ignatiuz offers..." in loops.
        # 1.15 = gentle penalty. 1.0 = no penalty. Values > 1.3 can distort output.
 
        stop = ["---DOCUMENT EXCERPT", "User:", "\n\nUser"],
        # List of strings that immediately stop generation if produced.
        # "---DOCUMENT EXCERPT" stops the model if it tries to echo the chunk
        # back into its response (a common failure mode with grounded prompts).
        # "User:" and "\n\nUser" stop it from hallucinating a follow-up turn.
    )
 
    response = result["choices"][0]["message"]["content"]
    # result is a dict in OpenAI ChatCompletion format:
    # {
    #   "choices": [
    #     {
    #       "message": {
    #         "role": "assistant",
    #         "content": "The actual response text here"
    #       },
    #       "finish_reason": "stop"   ← or "length" if max_tokens was hit
    #     }
    #   ],
    #   "usage": {"prompt_tokens": N, "completion_tokens": M, "total_tokens": P}
    # }
    # We navigate to choices[0]["message"]["content"] to get the plain text.
    # choices[0] because we only requested one completion (the default).
 
    # Post-processing: clean up any echoed excerpt the model may have included
    # before the stop token could fire (can happen on the first token).
    response = re.sub(
        r'---DOCUMENT EXCERPT.*',
        '',
        response,
        flags=re.DOTALL,
        # DOTALL makes . match newlines — removes everything from the excerpt
        # marker to the end of the string if it somehow slipped through.
    )
 
    return response.strip()
    # .strip() removes any leading/trailing whitespace or newlines from the
    # final response before it gets saved into the training dataset.

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9 — QUALITY FILTERING
# ─────────────────────────────────────────────────────────────────────────────
 
def is_quality_response(response: str, chunk_text: str) -> Tuple[bool, str]:
    """
    Check if a teacher-generated response is good enough to include
    in the training dataset. Returns (is_good: bool, reason: str).
 
    This is the QUALITY GATE — the most important step in the pipeline.
    A student trained on bad samples learns bad behaviour permanently.
    """
 
    if len(response.strip()) < 80:
        return False, "Response too short (< 80 chars) — likely a refusal or empty output"
        # Very short responses usually mean the teacher said "I cannot answer"
        # or "The excerpt doesn't contain this" (which we DO keep — see below)
        # or just failed to generate anything useful.
 
    if "i cannot" in response.lower() or "i don't have" in response.lower():
        return False, "Teacher refused to answer — rephrase the question"
        # Refusals mean the question template didn't fit the chunk well.
        # These are discarded — they add no training value.
 
    if "this information is not in the provided section" in response.lower():
        return True, "Valid fallback — teaches the student document boundaries"
        # This is DIFFERENT from a refusal. This is the teacher correctly
        # saying "the document doesn't say this." We KEEP these samples because
        # they teach the student not to hallucinate when asked about topics
        # outside the document's scope. Very valuable training signal.
 
    # Check that the response actually references something from the chunk.
    # We extract significant words from the chunk (length > 5 to avoid articles)
    # and check that at least some appear in the response.
    chunk_keywords = set(
        word.lower() for word in re.findall(r'\b[A-Za-z]{5,}\b', chunk_text)
    )
    # Find all words ≥ 5 chars from the chunk (strips common articles/prepositions).
 
    response_words = set(word.lower() for word in re.findall(r'\b[A-Za-z]{5,}\b', response))
    # Same for the response.
 
    overlap = chunk_keywords & response_words
    # Set intersection: words appearing in both chunk and response.
 
    if len(overlap) < 5:
        return False, f"Response doesn't reference chunk content (only {len(overlap)} shared words)"
        # If fewer than 5 meaningful words overlap, the teacher probably
        # answered from its general knowledge rather than the chunk.
        # This would produce a hallucinated training sample.
 
    # Check for obvious hallucination patterns.
    hallucination_patterns = [
        r'\d{4,}(?:\s*(?:hours|staff|FTE|employees|workers))(?!\s+(?:per|saved|reduced))',
        # Large numbers followed by staff/hours that aren't in a "saved/reduced" context
        # may be invented. The negative lookahead (?!...) exempts legitimate patterns
        # like "1000 staff-hours saved" which IS in the document.
    ]
    for pattern in hallucination_patterns:
        match = re.search(pattern, response)
        if match:
            # Verify the matched text actually appears in the chunk.
            matched_text = match.group(0)
            if matched_text not in chunk_text:
                return False, f"Potential hallucination detected: '{matched_text}' not in source chunk"
 
    return True, "Passes quality checks"

In [13]:
teacher = load_teacher()

llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


Teacher loaded: gpt-oss-20b-F16.gguf
Context window: 2048 tokens


AttributeError: 'int' object has no attribute 'n_gpu_layers'

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10 — MAIN DATASET GENERATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
 
def generate_dataset(pdf_path: str) -> List[Dict]:
    """
    Full pipeline: PDF + DOCUMENT_KNOWLEDGE_BASE → chunks → teacher Q&A → dataset.
    Returns a list of training samples in the messages format.
 
    TWO-SOURCE CHUNKING STRATEGY:
    This pipeline uses BOTH sources to build chunks:
    1. DOCUMENT_KNOWLEDGE_BASE (primary) — verified strings extracted directly from the
       PDF with exact metrics, client names, and case study figures preserved perfectly.
       These are injected first and given priority as grounding context.
    2. fitz PDF extraction (secondary) — catches any content not covered by the knowledge
       base entries (diagrams described as text, additional slides, footer content).
    Together these guarantee both ACCURACY (from the knowledge base) and COVERAGE (from fitz).
    """
 
    # ── Step A: Build chunks from DOCUMENT_KNOWLEDGE_BASE (primary source) ───
    print("\n[1/5] Building verified chunks from DOCUMENT_KNOWLEDGE_BASE...")
    kb_chunks = []
    for section_name, section_text in DOCUMENT_KNOWLEDGE_BASE.items():
        # Each section in DOCUMENT_KNOWLEDGE_BASE becomes one or more chunks.
        # Section text is the EXACT content from the PDF — zero hallucination risk.
        cleaned = section_text.strip()
        if not cleaned:
            continue
        # Split long sections into CHUNK_SIZE pieces with CHUNK_OVERLAP overlap
        # so the teacher prompt never exceeds the n_ctx window of 2048 tokens.
        start = 0
        part = 0
        while start < len(cleaned):
            end = start + CHUNK_SIZE
            chunk_text = cleaned[start:end]
            kb_chunks.append({
                "chunk_id":   f"kb_{section_name}_{part}",
                # kb_ prefix distinguishes knowledge-base chunks from fitz chunks.
                "text":       chunk_text,
                "source":     "knowledge_base",
                # Marks this chunk as coming from the verified knowledge base,
                # not from raw fitz extraction — used in audit logging.
                "section":    section_name,
                # Which DOCUMENT_KNOWLEDGE_BASE key this came from.
                "char_start": start,
                "char_end":   start + len(chunk_text),
            })
            start += len(chunk_text) - CHUNK_OVERLAP
            part += 1
    print(f"   Built {len(kb_chunks)} verified chunks from {len(DOCUMENT_KNOWLEDGE_BASE)} sections")
 
    # ── Step B: Extract additional content from PDF via fitz (secondary source) ──
    print("\n[2/5] Extracting supplementary content from PDF via fitz...")
    try:
        pages = extract_pages_with_fitz(pdf_path)
        # PyMuPDF extracts page-by-page text in reading order.
 
        tables = extract_tables_with_pdfplumber(pdf_path)
        # pdfplumber extracts table row/column structure.
 
        page_table_map = {}
        for t in tables:
            page_table_map.setdefault(t["page_num"], []).append(t["table_text"])
 
        for page in pages:
            if page["page_num"] in page_table_map:
                page["text"] += "\n\n[TABLE DATA]\n" + "\n\n".join(page_table_map[page["page_num"]])
 
        with open(OUTPUT_DIR / "extracted_pages.json", "w") as f:
            json.dump(pages, f, indent=2)
        print(f"   Saved extracted_pages.json ({len(pages)} pages)")
 
        fitz_chunks = chunk_pages(pages)
        # chunk_pages() splits the merged fitz text into overlapping windows.
 
        # Mark all fitz chunks so we can distinguish them in audit logs.
        for c in fitz_chunks:
            c["source"] = "fitz_extraction"
            c["section"] = "pdf_page"
 
        with open(OUTPUT_DIR / "fitz_chunks.json", "w") as f:
            json.dump(fitz_chunks, f, indent=2)
        print(f"   Saved fitz_chunks.json ({len(fitz_chunks)} fitz chunks)")
 
    except Exception as e:
        print(f"   Warning: fitz extraction failed ({e}). Using knowledge base only.")
        fitz_chunks = []
        # If the PDF can't be read (permissions, path issue), we fall back to
        # the knowledge base chunks alone — still a valid dataset.
 
    # Combine: knowledge base chunks first (priority), then fitz chunks.
    all_chunks = kb_chunks + fitz_chunks
    # Knowledge base chunks go first so the teacher sees verified content early
    # in the generation loop and the highest-quality samples are produced first.
 
    with open(OUTPUT_DIR / "all_chunks.json", "w") as f:
        json.dump(all_chunks, f, indent=2)
    print(f"   Total chunks: {len(all_chunks)} ({len(kb_chunks)} verified + {len(fitz_chunks)} fitz)")
 
    # ── Step C: Load the teacher model ────────────────────────────────────
    print("\n[3/5] Loading teacher model (gpt-oss-20b F16 GGUF)...")
    teacher = load_teacher()
    # Loads directly from your local WSL cache — no download needed.
 
    # ── Step D: Generate Q&A pairs from each chunk ────────────────────────
    print("\n[4/5] Generating dataset from chunks...")
    dataset = []
    rejected = 0
 
    for chunk_idx, chunk in enumerate(all_chunks):
        source_label = chunk.get("source", "unknown")
        print(f"\n  Chunk {chunk_idx + 1}/{len(all_chunks)} "
              f"[{source_label}] "
              f"({len(chunk['text'])} chars, "
              f"starts: '{chunk['text'][:50].strip()}...')")
 
        topic = extract_topic_from_chunk(chunk["text"])
        # Detect the main topic of this chunk for filling question templates.
 
        # ── Direct Q&A samples (70% of dataset target) ────────────────────
        selected_templates = random.sample(DIRECT_QUESTION_TEMPLATES,
                                           min(QA_PER_CHUNK, len(DIRECT_QUESTION_TEMPLATES)))
        # Randomly pick QA_PER_CHUNK (3) templates WITHOUT replacement.
        # random.sample ensures we never ask the same question twice for one chunk.
 
        for template in selected_templates:
            question = template.format(topic=topic)
            # Fill the {topic} placeholder with the detected topic keyword.
 
            messages = build_generation_prompt(chunk["text"], question, use_cot=False)
            # Build the grounded prompt with the chunk as context.
 
            response = generate_grounded_response(teacher, messages, use_cot=False)
            # Ask the teacher to answer, grounded in the chunk text.
 
            is_good, reason = is_quality_response(response, chunk["text"])
            # Run quality checks.
 
            if is_good:
                dataset.append({
                    "messages": [
                        {"role": "system",    "content": SYSTEM_PROMPT},
                        # Note: we use SYSTEM_PROMPT WITHOUT the chunk text here.
                        # The STUDENT'S training sample does NOT include the chunk —
                        # the student learns to answer from memory, not from context.
                        # The chunk was only used to make the TEACHER's answer accurate.
                        {"role": "user",      "content": question},
                        {"role": "assistant", "content": response},
                    ],
                    "_meta": {
                        "chunk_id": chunk["chunk_id"],
                        "type":     "direct",
                        "topic":    topic,
                    }
                    # _meta is a debug field — it's stripped before training
                    # but helps you trace each sample back to its source chunk
                    # if you need to audit or fix a bad sample.
                })
                print(f"    ✓ Direct: '{question[:55]}...'")
            else:
                rejected += 1
                print(f"    ✗ Rejected ({reason}): '{question[:55]}...'")
 
        # ── CoT samples (30% of dataset target) ────────────────────────────
        if chunk_idx % 2 == 0:
            # Only generate CoT for every other chunk (alternating).
            # Reason: CoT generation takes ~3–4x longer (more tokens).
            # With 40 chunks, generating CoT for all 40 would take too long
            # and over-represent CoT in the dataset. Every-other-chunk gives
            # a natural ~30% CoT ratio without careful counting.
 
            cot_template = random.choice(COT_SCENARIO_TEMPLATES)
            # Pick one random CoT scenario template.
 
            cot_question = cot_template.format(topic=topic)
            messages_cot = build_generation_prompt(chunk["text"], cot_question, use_cot=True)
            response_cot = generate_grounded_response(teacher, messages_cot, use_cot=True)
 
            is_good, reason = is_quality_response(response_cot, chunk["text"])
 
            if is_good:
                dataset.append({
                    "messages": [
                        {"role": "system",    "content": SYSTEM_PROMPT + "\nReasoning: high"},
                        {"role": "user",      "content": cot_question},
                        {"role": "assistant", "content": response_cot},
                    ],
                    "_meta": {
                        "chunk_id": chunk["chunk_id"],
                        "type":     "cot",
                        "topic":    topic,
                    }
                })
                print(f"    ✓ CoT:    '{cot_question[:55]}...'")
            else:
                rejected += 1
                print(f"    ✗ CoT rejected ({reason})")
 
    print(f"\n  Generation complete: {len(dataset)} samples kept, {rejected} rejected")
 
    # ── Step E: Save the dataset ───────────────────────────────────────────
    print("\n[5/5] Saving dataset...")
 
    # Remove _meta fields before saving (not needed for training).
    clean_dataset = [{"messages": s["messages"]} for s in dataset]
 
    random.shuffle(clean_dataset)
    # Shuffle to mix direct and CoT samples across the train/eval split.
 
    split = int(len(clean_dataset) * 0.9)
    train_data = clean_dataset[:split]
    eval_data  = clean_dataset[split:]
    # 90% / 10% train / eval split.
 
    def save_jsonl(data, path):
        with open(path, "w") as f:
            for item in data:
                f.write(json.dumps(item) + "\n")
                # One JSON object per line = JSONL format.
 
    save_jsonl(train_data, OUTPUT_DIR / "train.jsonl")
    save_jsonl(eval_data,  OUTPUT_DIR / "eval.jsonl")
 
    # Also save a debug version WITH _meta for auditing.
    with open(OUTPUT_DIR / "dataset_with_meta.json", "w") as f:
        json.dump(dataset, f, indent=2)
    # This lets you open dataset_with_meta.json, find a bad sample, and
    # trace it back to chunk_id to see exactly which part of the PDF
    # produced it.
 
    print(f"""
   Dataset saved to {OUTPUT_DIR}/
   ├── train.jsonl          ({len(train_data)} samples)
   ├── eval.jsonl           ({len(eval_data)} samples)
   ├── dataset_with_meta.json (full debug view)
   ├── chunks.json          ({len(chunks)} text chunks)
   └── extracted_pages.json ({len(pages)} pages)
    """)
 
    return clean_dataset

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 11 — DATASET AUDIT TOOL
# ─────────────────────────────────────────────────────────────────────────────
 
def audit_dataset(dataset_path: str, n_samples: int = 10):
    """
    Print n random samples from the dataset for human review.
    Run this BEFORE fine-tuning to catch any bad samples.
 
    This is the most important step most people skip.
    A 10-minute audit here can save hours of debugging why the student
    gives wrong answers after training.
    """
    with open(dataset_path) as f:
        samples = [json.loads(line) for line in f]
    # Load all lines of the JSONL file into a list of dicts.
 
    sampled = random.sample(samples, min(n_samples, len(samples)))
    # Pick n random samples.
 
    print(f"\n{'='*70}")
    print(f"DATASET AUDIT — {n_samples} random samples from {dataset_path}")
    print(f"{'='*70}")
 
    for i, sample in enumerate(sampled):
        print(f"\n--- Sample {i+1} ---")
        for msg in sample["messages"]:
            role = msg["role"].upper()
            content = msg["content"]
            if role == "SYSTEM":
                print(f"[SYSTEM]: {content[:120]}...")
                # Only show first 120 chars of system prompt (it's always the same).
            elif role == "USER":
                print(f"[USER]:   {content}")
            elif role == "ASSISTANT":
                print(f"[ASSISTANT]: {content[:500]}...")
                # Show first 500 chars of assistant response.
                # For CoT responses this shows the beginning of the reasoning trace.
 
        print()
        input("Press Enter for next sample (Ctrl+C to stop)...")
        # Interactive pause — lets you read each sample carefully before proceeding.

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 12 — ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
 
if __name__ == "__main__":
    # This block only runs when you execute the script directly:
    # python grounded_pipeline.py
    # It does NOT run when the file is imported as a module.
 
    print("=== DOCUMENT-GROUNDED DATASET GENERATION ===\n")
 
    # Step 1: Generate the dataset.
    dataset = generate_dataset(PDF_PATH)
    # Runs the full pipeline: extract → chunk → generate → filter → save.
 
    # Step 2: Audit before training.
    print("\n=== AUDITING TRAINING SET ===")
    audit_dataset(str(OUTPUT_DIR / "train.jsonl"), n_samples=10)
    # Review 10 random samples interactively.
    # Check: Do answers use exact numbers from the document?
    # Check: Do answers avoid adding information not in the chunk?
    # Check: Are CoT reasoning traces citing document evidence?
    # If you see problems: open dataset_with_meta.json, find the bad sample,
    # note its chunk_id, open chunks.json, find that chunk, and identify
    # which part of the PDF produced the bad output.
 
    print("\n=== READY FOR FINE-TUNING ===")
    print(f"Training file: {OUTPUT_DIR}/train.jsonl")
    print(f"Eval file:     {OUTPUT_DIR}/eval.jsonl")
    print("Next step: run your fine-tuning script pointing to these files.")

=== DOCUMENT-GROUNDED DATASET GENERATION ===


[1/5] Building verified chunks from DOCUMENT_KNOWLEDGE_BASE...


: 

In [1]:
torch.cuda.is_available()

NameError: name 'torch' is not defined

In [ ]:
n_gpu_layers=-1

In [6]:
!nvidia-smi 

Thu Mar 26 15:14:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.64.01              Driver Version: 576.88         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   39C    P8             35W /  600W |     482MiB /  32607MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
import time, psutil

# Clears unused GPU memory
# Frees cached tensors
# Prevents fake VRAM spikes
# Reset GPU stats

# peak GPU memory used
# This line resets peak counter so:
# You measure only THIS training run
# Not previous runs / notebooks

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [5]:

from llama_cpp import Llama

llm = Llama(
    model_path="/mnt/c/Users/Tron/.cache/huggingface/hub/models--unsloth--gpt-oss-20b-GGUF/snapshots/d449b42d93e1c2c7bda5312f5c25c8fb91dfa9b4/gpt-oss-20b-F16.gguf",
    n_gpu_layers=-1,
    n_ctx=2048,
)


llama_model_loader: loaded meta data with 37 key-value pairs and 459 tensors from /mnt/c/Users/Tron/.cache/huggingface/hub/models--unsloth--gpt-oss-20b-GGUF/snapshots/d449b42d93e1c2c7bda5312f5c25c8fb91dfa9b4/gpt-oss-20b-F16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gpt-oss
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Gpt-Oss-20B
llama_model_loader: - kv   3:                           general.basename str              = Gpt-Oss-20B
llama_model_loader: - kv   4:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   5:                         general.size_label str              = 20B
llama_model_loader: - kv   6:               

: 

In [16]:
from llama_cpp import Llama

teacher = Llama(
    model_path = "/mnt/c/Users/Tron/.cache/huggingface/hub/models--unsloth--gpt-oss-20b-GGUF/snapshots/d449b42d93e1c2c7bda5312f5c25c8fb91dfa9b4/gpt-oss-20b-F16.gguf",
    n_gpu_layers = -1,
    n_ctx        = 2048,
    verbose      = True,   # ← temporarily enable this
)

llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GeForce RTX 5090) (0000:01:00.0) - 18309 MiB free
llama_model_loader: loaded meta data with 37 key-value pairs and 459 tensors from /mnt/c/Users/Tron/.cache/huggingface/hub/models--unsloth--gpt-oss-20b-GGUF/snapshots/d449b42d93e1c2c7bda5312f5c25c8fb91dfa9b4/gpt-oss-20b-F16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gpt-oss
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Gpt-Oss-20B
llama_model_loader: - kv   3:                           general.basename str              = Gpt-Oss-20B
llama_model_loader: - kv   4:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   5: 